# CP2/CP3 — Generative AI for Engineering

**Tema:** LLM para diagnóstico e correção de erros em Python.  
**Modelo:** Qwen/Qwen2.5-0.5B-Instruct (pesos abertos, execução direta).  
**Método:** fine-tuning supervisionado com LoRA.  
**Recursos:** GPU CUDA do Google Colab (plano Free, sujeita à disponibilidade).

> **Antes de executar:** em **Ambiente de execução > Alterar o tipo de ambiente de execução**, selecione **GPU**. Sem CUDA, o notebook interrompe treino e avaliação intencionalmente. Não há resultados pré-fabricados.

**Objetivo:** comparar o modelo original, a configuração A e a configuração B no mesmo teste reservado; coletar métricas reais e gerar evidências.

## 1. Verificar GPU e CUDA

No VS Code conectado ao Colab, deixe esta célula e a saída visíveis para o print solicitado pelo professor. A CPU do notebook pessoal não precisa ter placa NVIDIA.

In [ ]:
import torch, subprocess, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA (build):", torch.version.cuda)
print("CUDA disponível:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Ative uma GPU em Ambiente de execução > Alterar tipo de ambiente de execução."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM total (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Tensor realmente criado na GPU:", torch.ones(3, device="cuda").device)
subprocess.run(["nvidia-smi"], check=True)

## 2. Obter o código do GitHub e instalar bibliotecas

Clone ou atualize a versão do repositório. O PyTorch com CUDA já vem nos runtimes GPU do Colab; **não substitua a instalação do CUDA manualmente no Colab**.

In [ ]:
import os, subprocess
from pathlib import Path
repo = Path("/content/GenerativeAI-Engineering")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/GuilhermeSSantos2004/GenerativeAI-Engineering.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
print("Pasta atual:", os.getcwd())
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 3. Testar integridade do dataset

Três splits independentes por identificação e texto exato: 96 treino, 16 validação e 16 teste. Atenção: são variações sintéticas de padrões de erro, não um benchmark de produção.

In [ ]:
subprocess.run(["python", "-m", "pytest", "-q", "tests"], check=True)
subprocess.run(["python", "scripts/check_gpu.py", "--output", "results/gpu.json"], check=True)

## 4. Baseline — respostas do modelo antes do fine-tuning

Inferência **na GPU** com temperatura zero, mesmas questões que serão usadas para as configurações treinadas. O baseline é executado antes do treinamento.

In [ ]:
subprocess.run(["python", "-m", "src.evaluate",
    "--test-file", "data/test.jsonl",
    "--output", "results/baseline.json",
    "--temperature", "0"], check=True)

## 5. Fine-tuning A — LoRA r=8, learning rate 2e-4, 2 épocas

Fine-tuning verdadeiro: ajustes nos pesos do adapter LoRA; base congelada. O script salva perdas de treino/validação, tempo, VRAM e checkpoint.

In [ ]:
subprocess.run(["python", "-m", "src.train",
    "--output-dir", "outputs/experiment_a",
    "--learning-rate", "0.0002", "--epochs", "2",
    "--batch-size", "2", "--grad-accum", "4", "--lora-r", "8",
    "--seed", "42"], check=True)

## 6. Avaliar o adapter A

In [ ]:
subprocess.run(["python", "-m", "src.evaluate",
    "--test-file", "data/test.jsonl",
    "--adapter", "outputs/experiment_a/final_adapter",
    "--output", "results/experiment_a.json",
    "--temperature", "0"], check=True)

## 7. Fine-tuning B — LoRA r=16, learning rate 5e-5, 3 épocas

O segundo experimento altera **r**, **learning rate** e **épocas**. O batch size permanece constante para comparação mais controlada, mas há mais de uma variável mudando; não atribua diferenças de desempenho a um único parâmetro.

In [ ]:
subprocess.run(["python", "-m", "src.train",
    "--output-dir", "outputs/experiment_b",
    "--learning-rate", "0.00005", "--epochs", "3",
    "--batch-size", "2", "--grad-accum", "4", "--lora-r", "16",
    "--seed", "42"], check=True)

## 8. Avaliar o adapter B

In [ ]:
subprocess.run(["python", "-m", "src.evaluate",
    "--test-file", "data/test.jsonl",
    "--adapter", "outputs/experiment_b/final_adapter",
    "--output", "results/experiment_b.json",
    "--temperature", "0"], check=True)

## 9. Comparação quantitativa e análise de respostas

Métricas: presença de seções, recuperação de palavras-chave e taxa de respostas com todos os termos esperados. **Não equivalem à avaliação de correção semântica**. Leia manualmente as respostas salvas nos JSON.

In [ ]:
subprocess.run(["python", "-m", "src.report", "--inputs",
    "results/baseline.json", "results/experiment_a.json",
    "results/experiment_b.json", "--output-dir", "results/summary"], check=True)
from IPython.display import Image, display
import pandas as pd
display(pd.read_csv("results/summary/comparison.csv"))
display(Image(filename="results/summary/comparison.png"))
for variant in ["baseline", "experiment_a", "experiment_b"]:
    import json
    result = json.loads(Path(f"results/{variant}.json").read_text(encoding="utf-8"))
    print(f"\n{variant} — Exemplo de resposta real no teste:")
    print(result["examples"][0]["generated_answer"][:1000])

## 10. Exportar evidências reais

Os arquivos em **evidences/** são cópias dos experimentos, podendo ser commitados após a execução. Os adapters permanecem em **outputs/** (não são commitados por padrão).

In [ ]:
subprocess.run(["python", "scripts/export_evidence.py", "--strict"], check=True)
subprocess.run(["python", "scripts/build_markdown_report.py"], check=True)
print("Arquivos de evidência prontos para serem baixados:", sorted(str(p) for p in Path("evidences").glob("*")))

## 11. Download de artefatos para guardar, revisar e publicar

Salve o ZIP e os prints **antes de encerrar o Colab**. Sessões gratuitas expiram e podem perder arquivos temporários. O ZIP inclui adapter e resultados, mas não inclui o modelo base da Hugging Face.

In [ ]:
import zipfile
zip_target = Path("/content/CP2_CP3_resultados_reais.zip")
with zipfile.ZipFile(zip_target, "w", zipfile.ZIP_DEFLATED) as archive:
    for root in ("results", "evidences", "outputs/experiment_a/final_adapter",
                 "outputs/experiment_b/final_adapter"):
        for file in Path(root).rglob("*"):
            if file.is_file():
                archive.write(file, arcname=str(file))
print("Criado:", zip_target, "bytes:", zip_target.stat().st_size)
from google.colab import files
files.download(str(zip_target))

## 12. Frontend Gradio para usuário inserir o prompt

Esta célula mantém o servidor rodando; coloque-a por último. `--share` gera link público temporário, use somente códigos e erros não confidenciais. Se preferir o experimento B, altere `--adapter`.

In [ ]:
subprocess.run(["python", "-m", "src.ui",
    "--adapter", "outputs/experiment_a/final_adapter",
    "--share"], check=True)